In [ ]:
import pandas as pd
import numpy as np
import sidrapy
from pathlib import Path

# Etapa de AQUISIÇÃO da soja: baixa a Tabela 1612 (PAM) e grava em
# data/raw/tabela1612.csv no formato de exportação do SIDRA, que é a
# entrada do main.ipynb (soja_clima.sidra.ler_sidra_soja).
RAW_DIR = Path("../../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import time

print("Baixando dados do SIDRA via API (Tabela 1612) em lotes por ano...")

# Lista de anos (2000 a 2025) - ajuste se necessário
anos = [str(ano) for ano in range(2000, 2026)]

lista_dfs = []

for ano in anos:
    print(f"  -> Baixando ano {ano}...")
    try:
        df_ano = sidrapy.get_table(
            table_code="1612",
            territorial_level="6", # Municípios
            ibge_territorial_code="all",
            variable="all",
            classifications={"81": "2713"}, # Soja (em grão)
            period=ano
        )
        # Remove a linha de cabeçalho que o sidrapy retorna
        df_ano = df_ano.iloc[1:].reset_index(drop=True)
        lista_dfs.append(df_ano)
        time.sleep(0.5) # Pausa para não sobrecarregar a API
    except Exception as e:
        print(f"     ⚠️ Erro no ano {ano}: {e}")
        continue

# Concatena todos os anos em um único DataFrame
df_api = pd.concat(lista_dfs, ignore_index=True)

# Renomeia as colunas para o padrão do projeto
df_api = df_api.rename(columns={
    'D1C': 'codigo_muni',
    'D2C': 'ano',
    'D3N': 'variavel', # <--- CORRIGIDO AQUI (era D4N)
    'V': 'valor'
})

# Limpa os dados
df_api['valor'] = pd.to_numeric(
    df_api['valor'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).replace(['-', '...', 'X'], np.nan),
    errors='coerce'
)
df_api = df_api.dropna(subset=['codigo_muni', 'ano', 'valor'])
df_api['ano'] = df_api['ano'].astype(int)
df_api['codigo_muni'] = df_api['codigo_muni'].astype(int).astype(str)

# Padroniza o nome das variáveis
df_api['variavel'] = (
    df_api['variavel']
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('ã', 'a')
    .str.replace('ç', 'c')
    .str.replace('é', 'e')
    .str.replace('ó', 'o')
)

print(f"\n✅ Download concluído! Total de linhas: {len(df_api)}")
print(f"Variáveis encontradas: {df_api['variavel'].unique()}")

In [ ]:
# Variáveis da Tabela 1612 no formato/cabeçalho do arquivo exportado pelo SIDRA
VARIAVEIS_SIDRA = {
    '109': 'Área plantada (Hectares)',
    '216': 'Área colhida (Hectares)',
    '214': 'Quantidade produzida (Toneladas)',
    '112': 'Rendimento médio da produção (Quilogramas por Hectare)',
}

tab = df_api[df_api['D3C'].isin(VARIAVEIS_SIDRA)].copy()
tab['coluna'] = tab['D3C'].map(VARIAVEIS_SIDRA)
largo = (tab.pivot_table(index=['codigo_muni', 'D1N', 'ano'], columns='coluna',
                         values='valor', aggfunc='first')
            .reset_index())
# 'Alta Floresta D'Oeste - RO' -> 'Alta Floresta D'Oeste (RO)' (formato do export)
largo['Município'] = largo['D1N'].str.replace(r'^(.*) - ([A-Z]{2})$', r'\1 (\2)', regex=True)
largo = largo.rename(columns={'ano': 'Ano'})[['Município', 'Ano', *VARIAVEIS_SIDRA.values()]]

caminho = RAW_DIR / "tabela1612.csv"
with open(caminho, 'w', encoding='utf-8-sig') as f:
    f.write('"Tabela 1612 - Soja (em grão) por município"\n""\n')
    largo.to_csv(f, sep=';', index=False)
    f.write('"Fonte: IBGE - Produção Agrícola Municipal"\n')

print(f"✅ {len(largo):,} linhas (município-ano) salvas em {caminho}")
print(f"   Produção 2020: {largo.loc[largo['Ano'] == 2020, 'Quantidade produzida (Toneladas)'].sum() / 1e6:.1f} Mt")
display(largo.head())